# 🧠 Chương 7: Deep Learning cho Time Series
## Advanced Data Science - Session 5

---

**Mục tiêu chương này:**
- Hiểu kiến trúc RNN, LSTM, GRU
- Xây dựng mô hình LSTM cho Time Series Forecasting
- 1D CNN cho Time Series
- CNN-LSTM Hybrid
- Encoder-Decoder cho Multi-step Forecasting
- So sánh Deep Learning vs Traditional Methods

## 7.1 RNN - Recurrent Neural Network

### 🎯 Ý tưởng:
Neural network thông thường xử lý mỗi input **độc lập**. Nhưng Time Series có **thứ tự** → Cần mạng **nhớ được quá khứ**.

**Ví dụ đời thường:**
- Đọc sách: Bạn hiểu câu hiện tại nhờ **nhớ** các câu trước đó
- RNN: Xử lý Y_t dựa trên **memory** từ Y_{t-1}, Y_{t-2}, ...

### Vấn đề: Vanishing Gradient
- RNN chỉ nhớ được **ngắn hạn** (5-10 steps)
- Gradient bị nhỏ dần qua nhiều timesteps → Quên quá khứ xa

## 7.2 LSTM - Long Short-Term Memory

### 🎯 Giải quyết Vanishing Gradient bằng 3 Gate:

| Gate | Chức năng | Ví dụ |
|------|-----------|-------|
| **Forget Gate** | Quên thông tin cũ không cần thiết | "Quên giá tuần trước, tập trung hôm nay" |
| **Input Gate** | Chọn thông tin mới để nhớ | "Ghi nhớ: Có event đặc biệt" |
| **Output Gate** | Quyết định output | "Dựa trên memory → Dự đoán" |

**Các biến thể LSTM:**
- **Vanilla LSTM:** 1 lớp LSTM
- **Stacked LSTM:** Nhiều lớp LSTM chồng lên nhau (deep)
- **Bidirectional LSTM:** Đọc cả 2 chiều (quá khứ + tương lai)

## 7.3 GRU - Gated Recurrent Unit

Phiên bản **đơn giản hóa** của LSTM:
- Chỉ có 2 gates (Reset + Update) thay vì 3
- Ít parameters hơn → Train nhanh hơn
- Hiệu quả tương đương LSTM trong nhiều trường hợp

## 7.4 1D CNN cho Time Series

- CNN (Convolutional) thường dùng cho ảnh
- **1D CNN** trượt filter qua chuỗi thời gian → Phát hiện **local patterns**
- Nhanh hơn LSTM, tốt cho **short-term patterns**

## 7.5 Data Preparation cho Deep Learning

⚠️ **Quan trọng:** Deep Learning cần data dạng 3D:
- Shape: (samples, timesteps, features)
- Ví dụ: Dùng 30 ngày trước → Dự đoán ngày tiếp theo
  - Input shape: (n_samples, 30, 1)
  - Output shape: (n_samples, 1)

---
## 🔬 Phần Thực Hành
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, 
    Conv1D, MaxPooling1D, Flatten,
    Bidirectional, Input, TimeDistributed,
    RepeatVector
)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

def evaluate(actual, predicted, model_name=''):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    print(f'📊 {model_name:25s} | MAE: {mae:.4f} | RMSE: {rmse:.4f}')
    return {'MAE': mae, 'RMSE': rmse}

print(f'✅ TensorFlow version: {tf.__version__}')

### 📝 Ví dụ 1: Chuẩn Bị Data cho Deep Learning

In [ ]:
# Tạo dữ liệu mẫu
np.random.seed(42)
n = 1000
t = np.arange(n)
trend = 0.01 * t
seasonal = 10 * np.sin(2 * np.pi * t / 50)
noise = np.random.normal(0, 2, n)
data = 100 + trend + seasonal + noise

print('='*60)
print('  DATA PREPARATION CHO DEEP LEARNING')
print('='*60)

# Bước 1: Scale data (MinMaxScaler)
print('\n📌 Bước 1: Scale Data (MinMaxScaler)')
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data.reshape(-1, 1))
print(f'   Original range: [{data.min():.2f}, {data.max():.2f}]')
print(f'   Scaled range:   [{data_scaled.min():.4f}, {data_scaled.max():.4f}]')
print('   💡 Neural Networks hoạt động tốt nhất với data trong [0, 1]')

# Bước 2: Tạo sequences
print('\n📌 Bước 2: Tạo Sequences (Sliding Window)')
def create_sequences(data, lookback=30):
    """Tạo input/output sequences cho LSTM
    
    Ví dụ lookback=3:
    [1,2,3,4,5,6] → X: [[1,2,3],[2,3,4],[3,4,5]] Y: [4,5,6]
    """
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i-lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

LOOKBACK = 30  # Dùng 30 ngày trước để dự đoán
X, y = create_sequences(data_scaled, LOOKBACK)

print(f'   Lookback window: {LOOKBACK} timesteps')
print(f'   X shape: {X.shape} (samples, timesteps)')
print(f'   y shape: {y.shape} (samples,)')

# Bước 3: Reshape cho LSTM (3D)
print('\n📌 Bước 3: Reshape cho LSTM')
X = X.reshape(X.shape[0], X.shape[1], 1)  # (samples, timesteps, features)
print(f'   X shape after reshape: {X.shape}')
print(f'   → ({X.shape[0]} samples, {X.shape[1]} timesteps, {X.shape[2]} feature)')

# Bước 4: Train/Test split
print('\n📌 Bước 4: Train/Test Split')
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'   Train: {X_train.shape[0]} samples')
print(f'   Test:  {X_test.shape[0]} samples')

# Visualize một sequence
fig, ax = plt.subplots(figsize=(14, 4))
idx = 100
ax.plot(range(LOOKBACK), X[idx, :, 0], 'b-o', markersize=3, label=f'Input ({LOOKBACK} steps)')
ax.plot(LOOKBACK, y[idx], 'r*', markersize=15, label='Target (predict this)')
ax.axvline(x=LOOKBACK-0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_title('📊 Minh Họa: Input Sequence → Target', fontweight='bold')
ax.set_xlabel('Timestep')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Ví dụ 2: Vanilla LSTM

In [ ]:
# ===== VANILLA LSTM =====
print('='*60)
print('  VANILLA LSTM')
print('='*60)

model_lstm = Sequential([
    LSTM(64, input_shape=(LOOKBACK, 1), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_lstm.compile(optimizer=Adam(0.001), loss='mse')
model_lstm.summary()

print('\n💡 Giải thích kiến trúc:')
print('  LSTM(64): 64 units nhớ, input = 30 timesteps × 1 feature')
print('  Dropout(0.2): Ngẫu nhiên tắt 20% neurons → Chống overfitting')
print('  Dense(32): Fully connected layer')
print('  Dense(1): Output = 1 giá trị dự đoán')

In [ ]:
# Train LSTM
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

# Plot training
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_lstm.history['loss'], label='Train Loss')
ax.plot(history_lstm.history['val_loss'], label='Val Loss')
ax.set_title('📊 LSTM Training History', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.show()

# Predict & Inverse scale
lstm_pred_scaled = model_lstm.predict(X_test, verbose=0)
lstm_pred = scaler.inverse_transform(lstm_pred_scaled).flatten()
y_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

evaluate(y_actual, lstm_pred, 'Vanilla LSTM')

print(f'\n💡 EarlyStopping dừng ở epoch {len(history_lstm.history["loss"])}')
print('💡 Val Loss không giảm nữa → Dừng để tránh overfitting')

### 📝 Ví dụ 3: Stacked LSTM

In [ ]:
# ===== STACKED LSTM =====
print('='*60)
print('  STACKED LSTM (Deep LSTM)')
print('='*60)

model_stacked = Sequential([
    LSTM(64, input_shape=(LOOKBACK, 1), return_sequences=True),  # ← return_sequences=True!
    Dropout(0.2),
    LSTM(32, return_sequences=False),  # Lớp cuối: return_sequences=False
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model_stacked.compile(optimizer=Adam(0.001), loss='mse')

print('💡 return_sequences=True: Output TOÀN BỘ sequence → Đưa vào LSTM tiếp theo')
print('💡 return_sequences=False: Output CHỈ timestep cuối → Đưa vào Dense')
print(f'   Total params: {model_stacked.count_params():,}')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_stacked = model_stacked.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop], verbose=0
)

stacked_pred = scaler.inverse_transform(model_stacked.predict(X_test, verbose=0)).flatten()
evaluate(y_actual, stacked_pred, 'Stacked LSTM')

### 📝 Ví dụ 4: GRU

In [ ]:
# ===== GRU =====
print('='*60)
print('  GRU (Gated Recurrent Unit)')
print('='*60)

model_gru = Sequential([
    GRU(64, input_shape=(LOOKBACK, 1), return_sequences=True),
    Dropout(0.2),
    GRU(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model_gru.compile(optimizer=Adam(0.001), loss='mse')
print(f'GRU params: {model_gru.count_params():,} (ít hơn LSTM!)')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_gru = model_gru.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop], verbose=0
)

gru_pred = scaler.inverse_transform(model_gru.predict(X_test, verbose=0)).flatten()
evaluate(y_actual, gru_pred, 'GRU')

print('\n💡 GRU chỉ có 2 gates (Reset + Update) thay vì 3 gates của LSTM')
print('💡 Ít parameters hơn → Train nhanh hơn')
print('💡 Hiệu quả tương đương LSTM trong nhiều trường hợp')

### 📝 Ví dụ 5: 1D CNN cho Time Series

In [ ]:
# ===== 1D CNN =====
print('='*60)
print('  1D CNN (Convolutional Neural Network)')
print('='*60)

model_cnn = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(LOOKBACK, 1)),
    Conv1D(filters=32, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model_cnn.compile(optimizer=Adam(0.001), loss='mse')

print('💡 Conv1D: Trượt filter qua chuỗi thời gian')
print('   kernel_size=3: Mỗi lần xét 3 timesteps liên tiếp')
print('   → Phát hiện local patterns (short-term)')
print(f'   Total params: {model_cnn.count_params():,}')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop], verbose=0
)

cnn_pred = scaler.inverse_transform(model_cnn.predict(X_test, verbose=0)).flatten()
evaluate(y_actual, cnn_pred, '1D CNN')

### 📝 Ví dụ 6: CNN-LSTM Hybrid

In [ ]:
# ===== CNN-LSTM HYBRID =====
print('='*60)
print('  CNN-LSTM HYBRID')
print('='*60)

model_cnn_lstm = Sequential([
    Conv1D(64, kernel_size=3, activation='relu', input_shape=(LOOKBACK, 1)),
    MaxPooling1D(pool_size=2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model_cnn_lstm.compile(optimizer=Adam(0.001), loss='mse')

print('💡 CNN-LSTM: CNN phát hiện local patterns → LSTM học long-term dependencies')
print('💡 Kết hợp ưu điểm cả 2: Nhanh (CNN) + Nhớ xa (LSTM)')
print(f'   Total params: {model_cnn_lstm.count_params():,}')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history_hybrid = model_cnn_lstm.fit(
    X_train, y_train,
    epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop], verbose=0
)

hybrid_pred = scaler.inverse_transform(model_cnn_lstm.predict(X_test, verbose=0)).flatten()
evaluate(y_actual, hybrid_pred, 'CNN-LSTM Hybrid')

### 📝 Ví dụ 7: So Sánh Tất Cả Deep Learning Models

In [ ]:
# ===== SO SÁNH =====
print('\n' + '='*70)
print('  BẢNG SO SÁNH TẤT CẢ DEEP LEARNING MODELS')
print('='*70)

all_preds = {
    'Vanilla LSTM': lstm_pred,
    'Stacked LSTM': stacked_pred,
    'GRU': gru_pred,
    '1D CNN': cnn_pred,
    'CNN-LSTM': hybrid_pred
}

results = {}
print('\n📊 Results:')
for name, pred in all_preds.items():
    results[name] = evaluate(y_actual, pred, name)

best = min(results.items(), key=lambda x: x[1]['RMSE'])
print(f'\n🏆 Best Model: {best[0]} (RMSE = {best[1]["RMSE"]:.4f})')

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: All predictions
test_range = range(len(y_actual))
axes[0].plot(test_range, y_actual, label='Actual', color='black', linewidth=2)
colors = ['blue', 'green', 'orange', 'red', 'purple']
for (name, pred), color in zip(all_preds.items(), colors):
    axes[0].plot(test_range, pred, label=name, color=color, alpha=0.7)
axes[0].legend(loc='upper left')
axes[0].set_title('📊 So Sánh Predictions', fontweight='bold', fontsize=14)

# Bottom: Zoomed in (last 50)
last_n = 50
axes[1].plot(range(last_n), y_actual[-last_n:], label='Actual', color='black', linewidth=2, marker='o', markersize=3)
for (name, pred), color in zip(all_preds.items(), colors):
    axes[1].plot(range(last_n), pred[-last_n:], label=name, color=color, alpha=0.7)
axes[1].legend(loc='upper left')
axes[1].set_title(f'📊 Zoom In (Last {last_n} steps)', fontweight='bold')

plt.tight_layout()
plt.show()

# RMSE comparison bar chart
fig, ax = plt.subplots(figsize=(10, 4))
names = list(results.keys())
rmses = [results[n]['RMSE'] for n in names]
bars = ax.bar(names, rmses, color=colors)
ax.set_title('📊 RMSE Comparison', fontweight='bold')
ax.set_ylabel('RMSE')
for bar, rmse in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{rmse:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

### 📝 Ví dụ 8: Encoder-Decoder cho Multi-step Forecasting

In [ ]:
# ===== ENCODER-DECODER =====
print('='*60)
print('  ENCODER-DECODER: Multi-step Forecasting')
print('='*60)

# Dự đoán 7 bước tiếp theo thay vì 1
FORECAST_STEPS = 7

def create_multi_step_sequences(data, lookback=30, forecast=7):
    X, y = [], []
    for i in range(lookback, len(data) - forecast + 1):
        X.append(data[i-lookback:i, 0])
        y.append(data[i:i+forecast, 0])
    return np.array(X), np.array(y)

X_multi, y_multi = create_multi_step_sequences(data_scaled, LOOKBACK, FORECAST_STEPS)
X_multi = X_multi.reshape(X_multi.shape[0], X_multi.shape[1], 1)

print(f'Input shape:  {X_multi.shape} (samples, {LOOKBACK} timesteps, 1 feature)')
print(f'Output shape: {y_multi.shape} (samples, {FORECAST_STEPS} forecast steps)')

# Split
split = int(len(X_multi) * 0.8)
X_train_m, X_test_m = X_multi[:split], X_multi[split:]
y_train_m, y_test_m = y_multi[:split], y_multi[split:]

# Encoder-Decoder model
model_ed = Sequential([
    # Encoder
    LSTM(64, input_shape=(LOOKBACK, 1), return_sequences=False),
    # Bridge
    RepeatVector(FORECAST_STEPS),
    # Decoder
    LSTM(64, return_sequences=True),
    TimeDistributed(Dense(1))
])

model_ed.compile(optimizer=Adam(0.001), loss='mse')

print('\n💡 Kiến trúc Encoder-Decoder:')
print('   Encoder LSTM: Nén input sequence → Context vector')
print('   RepeatVector: Lặp context vector → N lần')
print('   Decoder LSTM: Giải nén → Dự đoán N steps')
print('   TimeDistributed: Áp dụng Dense cho mỗi timestep')

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
y_train_m_3d = y_train_m.reshape(y_train_m.shape[0], y_train_m.shape[1], 1)
y_test_m_3d = y_test_m.reshape(y_test_m.shape[0], y_test_m.shape[1], 1)

model_ed.fit(
    X_train_m, y_train_m_3d,
    epochs=100, batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop], verbose=0
)

# Predict
ed_pred = model_ed.predict(X_test_m, verbose=0).reshape(-1, FORECAST_STEPS)

# Visualize multi-step prediction
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for idx, ax in enumerate(axes.flatten()):
    sample_idx = idx * (len(X_test_m) // 4)
    actual = y_test_m[sample_idx]
    predicted = ed_pred[sample_idx]
    
    ax.plot(range(FORECAST_STEPS), actual, 'b-o', label='Actual', markersize=5)
    ax.plot(range(FORECAST_STEPS), predicted, 'r--s', label='Predicted', markersize=5)
    ax.set_title(f'Sample {sample_idx}', fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('Step')

plt.suptitle(f'📊 Encoder-Decoder: {FORECAST_STEPS}-Step Forecast', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# Overall RMSE
rmse = np.sqrt(mean_squared_error(y_test_m.flatten(), ed_pred.flatten()))
print(f'\n📊 Encoder-Decoder {FORECAST_STEPS}-step RMSE (scaled): {rmse:.4f}')

---
## 🏋️ BÀI TẬP THỰC HÀNH
---

### Bài 1: LSTM cơ bản (⭐ Dễ)

1. Đọc dữ liệu `retail_sales_dataset.csv`
2. Tiền xử lý: resample monthly, MinMaxScaler
3. Tạo sequences với lookback=6
4. Xây LSTM đơn giản (1 layer, 32 units)
5. Train và đánh giá

In [ ]:
# TODO: Viết code ở đây

### Bài 2: So sánh DL Models (⭐⭐ Trung bình)

1. Xây dựng 4 models: LSTM, GRU, 1D CNN, CNN-LSTM
2. Train tất cả trên cùng data
3. So sánh RMSE, MAE
4. Vẽ biểu đồ so sánh
5. Nhận xét: Model nào tốt nhất? Tại sao?

In [ ]:
# TODO: Viết code ở đây

### Bài 3: Bidirectional LSTM + Hyperparameter Tuning (⭐⭐⭐ Nâng cao)

1. Xây Bidirectional LSTM
2. Thử các giá trị khác nhau:
   - lookback: 7, 14, 30
   - LSTM units: 32, 64, 128
   - learning_rate: 0.01, 0.001, 0.0001
3. Tìm tổ hợp best
4. So sánh với ARIMA/SARIMA từ Chương 5

In [ ]:
# TODO: Viết code ở đây

---
# 🧠 PHẦN ÔN TẬP & CỦNG CỐ KIẾN THỨC - CHƯƠNG 7

---

## 1️⃣ Nguyên Lí 80/20: 20% Kiến Thức Cốt Lõi Mang Lại 80% Giá Trị

> **Nếu bạn chỉ nhớ được 3 điều từ chương này, hãy nhớ:**

### 🔑 Kiến thức #1: LSTM giải quyết Vanishing Gradient bằng 3 Gate
| Gate | Chức năng | Quyết định |
|---|---|---|
| **Forget Gate** | Quên thông tin cũ | "Xóa giá tuần trước, không còn relevant" |
| **Input Gate** | Nhớ thông tin mới | "Ghi nhớ: có sự kiện đặc biệt hôm nay" |
| **Output Gate** | Xuất kết quả | "Dựa trên memory hiện tại → Dự đoán" |

- **RNN thường:** Chỉ nhớ 5-10 bước → Quên quá khứ xa
- **LSTM:** Nhớ được hàng trăm bước nhờ cell state "đường cao tốc"
- **GRU:** Phiên bản lite của LSTM (2 gates), train nhanh hơn, hiệu quả tương đương

### 🔑 Kiến thức #2: Data phải là 3D — (samples, timesteps, features)
```python
# Input shape cho Deep Learning Time Series:
X.shape = (n_samples, lookback, n_features)

# Ví dụ: 1000 mẫu, mỗi mẫu dùng 30 ngày trước, 1 biến
X.shape = (1000, 30, 1)

# QUAN TRỌNG: Phải scale data trước!
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)
```

### 🔑 Kiến thức #3: LSTM cơ bản chỉ cần ~10 dòng code
```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(50, input_shape=(lookback, 1)),  # 50 units
    Dense(1)                                # Output 1 giá trị
])
model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=50, batch_size=32)
```

> **Một câu tóm tắt:** *"LSTM nhớ dài hạn nhờ 3 gates, data phải 3D + scaled, và GRU là phiên bản nhanh hơn của LSTM."*

---

## 2️⃣ Mô Hình Hóa Kiến Thức: Phép Ẩn Dụ "Bộ Não Con Người"

### 🧠 Câu chuyện: LSTM giống cách BỘ NÃO xử lý ký ức

Hãy tưởng tượng bộ não bạn đọc một cuốn tiểu thuyết dài:

**RNN (Bộ nhớ ngắn hạn):** 🐟 Trí nhớ cá vàng
- Đọc trang 300, quên hết trang 1-100
- Chỉ nhớ vài trang gần đây
- → **Vanishing Gradient:** Thông tin bị "bay hơi" qua nhiều bước

**LSTM (Bộ nhớ dài hạn):** 🧠 Trí nhớ con người thực tế
- **Forget Gate = Quên có chọn lọc:**
  - "Nhân vật phụ chương 1 không quan trọng → QUÊN" ← Forget Gate mở
  - "Nhân vật chính → GIỮ LẠI" ← Forget Gate đóng
  
- **Input Gate = Ghi nhớ có chọn lọc:**
  - "Twist mới ở chương 15 cực quan trọng → GHI NHỚ" ← Input Gate mở
  - "Mô tả phong cảnh dài dòng → bỏ qua" ← Input Gate đóng
  
- **Output Gate = Trả lời có chọn lọc:**
  - Ai hỏi "Ai là hung thủ?" → Lục memory, chọn thông tin phù hợp → Trả lời
  - → Output gate quyết định "dùng phần nào của memory"

**GRU = Phiên bản "speed reading":**
- Gộp Forget + Input thành 1 bước → Nhanh hơn
- Hiệu quả tương đương LSTM cho hầu hết bài toán

### 🎨 So sánh kiến trúc:
```
  RNN (Cá vàng):          LSTM (Não người):         GRU (Speed reader):
  
  Input → [Hidden] → Out  Input → [F][I][O] → Out   Input → [R][U] → Out
           ↓                      ↓ ↓ ↓                     ↓ ↓
         Quên nhanh        Cell State (Highway)       Gộp Forget+Input
                           = Trí nhớ dài hạn         = Nhanh hơn
  
  Nhớ ~5-10 bước          Nhớ ~100+ bước             Nhớ ~100+ bước
  Train nhanh              Train chậm nhất            Train nhanh hơn LSTM
  Accuracy thấp            Accuracy cao nhất          Accuracy ≈ LSTM
```

### 🖼️ **1D CNN giống MẮT quét qua chuỗi:**
```
  Time Series: [10, 12, 15, 18, 20, 22, 25]
  
  CNN Filter (size=3):  [▓▓▓]
  Quét:                 [▓▓▓]         → Detect pattern [10,12,15]
                         [▓▓▓]        → Detect pattern [12,15,18]
                          [▓▓▓]       → Detect pattern [15,18,20]
                           ...
  
  → Phát hiện LOCAL PATTERNS (xu hướng ngắn hạn)
  → Nhanh hơn LSTM cho short-term patterns!
```

---

## 3️⃣ Liên Tưởng Với Cuộc Sống

### 📱 Gợi ý từ khi gõ điện thoại (RNN/LSTM)
- Bạn gõ: "Tôi muốn đi" → Gợi ý: "ăn", "chơi", "ngủ"
- **RNN:** Chỉ nhìn 2-3 từ trước → Gợi ý chung chung
- **LSTM:** Nhớ cả câu trước đó "Hôm nay mệt quá" → Gợi ý: "ngủ" (thông minh hơn!)

### 🎬 Netflix gợi ý phim (LSTM memory)
- **Forget Gate:** Quên phim bạn xem 5 năm trước (sở thích đã thay đổi)
- **Input Gate:** Ghi nhớ bạn vừa xem 3 phim kinh dị liên tiếp → "Thích kinh dị!"
- **Output Gate:** Gợi ý phim kinh dị mới, không gợi ý phim tình cảm

### 📸 Nhận diện khuôn mặt (CNN)
- CNN quét ảnh bằng filter → Phát hiện cạnh, mắt, mũi
- **1D CNN cho Time Series:** Quét chuỗi số bằng filter → Phát hiện xu hướng tăng, giảm, dao động
- Giống cách bạn **lướt mắt qua biểu đồ** và thấy "đoạn này tăng, đoạn này giảm"

### 🎵 Sáng tác nhạc AI (Sequence prediction)
- AI học từ hàng nghìn bài hát: nốt trước → nốt tiếp theo
- **LSTM:** Nhớ giai điệu chủ đề ở đầu bài → Lặp lại ở cuối (long-term memory)
- **RNN:** Chỉ nhớ vài nốt → Giai điệu không mạch lạc

---

## 4️⃣ Teach to Learn — Giảng Lại Để Hiểu Sâu

### 📝 Bài tập 1: Giải thích LSTM bằng "câu chuyện đời thường"
> Dùng ví dụ **đọc truyện dài** để giải thích 3 gates của LSTM.
> Giải thích tại sao RNN "quên" (vanishing gradient) và LSTM "nhớ" được.

In [ ]:
# ✍️ BÀI TẬP 1: Giải thích LSTM bằng "câu chuyện đời thường"
# Dùng ví dụ đọc truyện dài để giải thích 3 gates

"""
CÂU CHUYỆN ĐỌC TRUYỆN DÀI:

Forget Gate (Quên) = _______________
  Ví dụ: _______________

Input Gate (Ghi nhớ) = _______________
  Ví dụ: _______________

Output Gate (Xuất kết quả) = _______________
  Ví dụ: _______________

TẠI SAO RNN "QUÊN" (Vanishing Gradient)?
→ _______________

TẠI SAO LSTM "NHỚ" ĐƯỢC?
→ _______________
"""

In [ ]:
# ✍️ BÀI TẬP 2: So sánh các kiến trúc Deep Learning
# Viết bảng so sánh RNN vs LSTM vs GRU vs 1D CNN

"""
BẢNG SO SÁNH:

| Tiêu chí         | RNN       | LSTM       | GRU        | 1D CNN     |
|------------------|-----------|------------|------------|------------|
| Số gates          | _________ | __________ | __________ | __________ |
| Memory length     | _________ | __________ | __________ | __________ |
| Tốc độ train      | _________ | __________ | __________ | __________ |
| Khi nào dùng?     | _________ | __________ | __________ | __________ |
| Ưu điểm chính     | _________ | __________ | __________ | __________ |
| Nhược điểm chính  | _________ | __________ | __________ | __________ |

KHI NÀO DÙNG CNN-LSTM HYBRID?
→ _______________

KHI NÀO DÙNG BIDIRECTIONAL LSTM?
→ _______________
"""

In [ ]:
# ✍️ BÀI TẬP 3: Viết code LSTM từ trí nhớ (không xem lại bài)
# Thử viết lại pipeline LSTM cơ bản từ đầu

"""
THỬ THÁCH: Viết code LSTM prediction từ trí nhớ!
Gợi ý các bước:
1. Import libraries
2. Scale data (MinMaxScaler)
3. Tạo sequences (lookback window)
4. Build LSTM model
5. Train
6. Predict & inverse transform
"""

# Bước 1: Import
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import LSTM, Dense
# from sklearn.preprocessing import MinMaxScaler
# import numpy as np

# Bước 2: Scale data
# scaler = MinMaxScaler()
# data_scaled = scaler.fit_transform(data.values.reshape(-1,1))

# Bước 3: Tạo sequences
# def create_sequences(data, lookback):
#     X, y = [], []
#     for i in range(___):  # Điền vào
#         X.append(data[___:___])  # Điền vào
#         y.append(data[___])  # Điền vào
#     return np.array(X), np.array(y)

# Bước 4: Build model
# model = Sequential([
#     LSTM(___, input_shape=(___, ___)),  # Điền units, timesteps, features
#     Dense(___)  # Điền output size
# ])
# model.compile(optimizer='___', loss='___')  # Điền optimizer, loss

# Bước 5: Train
# model.fit(X_train, y_train, epochs=___, batch_size=___)

# Bước 6: Predict
# predictions = model.predict(X_test)
# predictions_original = scaler.inverse_transform(predictions)